# ML: Delivery Time Prediction

> **Status: OPTIONAL / EXPERIMENTAL.** The model uses chronological evaluation,
> held-out interval calibration, and inference only for open ARRIVED shipments.

Predicts dwell time for shipments that have arrived but have no matched departure.

## Trustworthy Modeling Contract
- Features are limited to values known at arrival plus outcomes completed earlier
- Train, calibration, and test partitions are chronological
- Prediction intervals are calibrated on a held-out calibration partition
- Gold inference output excludes departure time, actual dwell, and prediction error

## Data Flow
```
Bronze truck arrival/departure state + dimensions --> Spark ML GBT --> Gold dwell_predictions
```

## Intended Use
Experimental dock-planning support. Do not automate operational decisions without
Fabric validation, monitoring, and a separately approved promotion gate.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from datetime import datetime, timedelta, timezone
import mlflow
import os

In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================

def get_env(var_name, default=None):
    return os.environ.get(var_name, default)

LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", default="retail_lakehouse")
BRONZE_SCHEMA = get_env("BRONZE_SCHEMA", default="cusn")
SILVER_DB = get_env("SILVER_DB", default="ag")
GOLD_DB = get_env("GOLD_DB", default="au")
EXPERIMENT_NAME = get_env("MLFLOW_EXPERIMENT", default="delivery_prediction")
TRUCK_ARRIVED_TABLE = get_env("TRUCK_ARRIVED_TABLE", default="truck_arrived")
TRUCK_DEPARTED_TABLE = get_env("TRUCK_DEPARTED_TABLE", default="truck_departed")
DIM_TRUCKS_TABLE = get_env("DIM_TRUCKS_TABLE", default="dim_trucks")
DIM_STORES_TABLE = get_env("DIM_STORES_TABLE", default="dim_stores")
DIM_DCS_TABLE = get_env("DIM_DCS_TABLE", default="dim_distribution_centers")
OUTPUT_TABLE = get_env("OUTPUT_TABLE", default="dwell_predictions")
MODEL_ARTIFACT_NAME = get_env("MODEL_ARTIFACT_NAME", default="gbt_dwell_model")
RUN_NAME_PREFIX = get_env("MLFLOW_RUN_PREFIX", default="gbt_dwell")

TEST_SIZE = float(get_env("TEST_SIZE", default="0.2"))
CALIBRATION_SIZE = float(get_env("CALIBRATION_SIZE", default="0.1"))
SPLIT_PURGE_HOURS = int(get_env("SPLIT_PURGE_HOURS", default="24"))
TARGET_MAE = float(get_env("TARGET_MAE", default="15.0"))
TARGET_MAPE = float(get_env("TARGET_MAPE", default="0.25"))
MAX_VALID_DWELL_MINUTES = float(get_env("MAX_VALID_DWELL_MINUTES", default="480"))
INTERVAL_COVERAGE = float(get_env("INTERVAL_COVERAGE", default="0.80"))
RANDOM_SEED = int(get_env("RANDOM_SEED", default="42"))

GBT_MAX_ITER = int(get_env("GBT_MAX_ITER", default="120"))
GBT_MAX_DEPTH = int(get_env("GBT_MAX_DEPTH", default="5"))
GBT_STEP_SIZE = float(get_env("GBT_STEP_SIZE", default="0.05"))
GBT_SUBSAMPLING_RATE = float(get_env("GBT_SUBSAMPLING_RATE", default="0.8"))
GBT_MAX_BINS = int(get_env("GBT_MAX_BINS", default="64"))

if not 0 < TEST_SIZE < 1:
    raise ValueError("TEST_SIZE must be between 0 and 1")
if not 0 < CALIBRATION_SIZE < 1:
    raise ValueError("CALIBRATION_SIZE must be between 0 and 1")
if TEST_SIZE + CALIBRATION_SIZE >= 1:
    raise ValueError("TEST_SIZE + CALIBRATION_SIZE must be less than 1")
if not 0 < INTERVAL_COVERAGE < 1:
    raise ValueError("INTERVAL_COVERAGE must be between 0 and 1")

LOWER_RESIDUAL_QUANTILE = (1.0 - INTERVAL_COVERAGE) / 2.0
UPPER_RESIDUAL_QUANTILE = 1.0 - LOWER_RESIDUAL_QUANTILE

print(f"Configuration: BRONZE_SCHEMA={BRONZE_SCHEMA}, SILVER_DB={SILVER_DB}, GOLD_DB={GOLD_DB}")
print(f"Lifecycle sources: {TRUCK_ARRIVED_TABLE}, {TRUCK_DEPARTED_TABLE}")
print(f"Dimension sources: {DIM_TRUCKS_TABLE}, {DIM_STORES_TABLE}, {DIM_DCS_TABLE}")
print(f"Output table: {OUTPUT_TABLE}")
print(f"Chronological partitions: calibration={CALIBRATION_SIZE:.1%}, test={TEST_SIZE:.1%}")
print(f"Model Targets: MAE < {TARGET_MAE} min, MAPE < {TARGET_MAPE*100}%")
print(f"Empirical interval coverage target: {INTERVAL_COVERAGE*100:.1f}%")

# Physical ML contract used by repository validation and the pre-write gate.
ML_SOURCE_TABLES = ('truck_arrived', 'truck_departed', 'dim_trucks', 'dim_stores', 'dim_distribution_centers')
ML_OUTPUT_CONTRACTS = {'dwell_predictions': [('shipment_id', 'string', False),
                       ('arrived_ts', 'timestamp', False),
                       ('location_type', 'string', False),
                       ('arrival_hour', 'int', False),
                       ('arrival_day_of_week', 'int', False),
                       ('predicted_dwell_minutes', 'double', False),
                       ('lower_bound_minutes', 'double', False),
                       ('upper_bound_minutes', 'double', False),
                       ('interval_coverage_target', 'double', False),
                       ('model_run_id', 'string', False),
                       ('prediction_status', 'string', False),
                       ('prediction_timestamp', 'timestamp', False)]}


In [ ]:
# =============================================================================
# HELPER FUNCTIONS & MLFLOW SETUP
# =============================================================================

def ensure_database(name):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {LAKEHOUSE_NAME}.{name}")
    print(f"Database '{name}' ready.")

def read_silver(table_name):
    return spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")

def read_bronze(table_name):
    return spark.table(f"{LAKEHOUSE_NAME}.{BRONZE_SCHEMA}.{table_name}")

def bronze_exists(table_name):
    try:
        spark.table(f"{LAKEHOUSE_NAME}.{BRONZE_SCHEMA}.{table_name}")
        return True
    except AnalysisException:
        return False

def silver_exists(table_name):
    try:
        spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")
        return True
    except AnalysisException:
        return False

def save_gold(df, table_name):
    full_name = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{table_name}"
    df.write.format("delta").mode("overwrite").saveAsTable(full_name)
    print(f"  {full_name}: {df.count()} rows")

def pick_existing_column(df, candidate_columns, label):
    columns_by_lower = {column.lower(): column for column in df.columns}
    for candidate in candidate_columns:
        resolved = columns_by_lower.get(candidate.lower())
        if resolved is not None:
            return resolved
    raise RuntimeError(
        f"Unable to find {label}. Checked columns: {', '.join(candidate_columns)}. "
        f"Available columns: {', '.join(df.columns)}"
    )

def resolve_table_column(df, table_name, *candidates):
    columns_by_lower = {column.lower(): column for column in df.columns}
    for candidate in candidates:
        resolved = columns_by_lower.get(candidate.lower())
        if resolved is not None:
            return resolved
    raise RuntimeError(
        f"Unable to resolve any of {candidates} in {LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}. "
        f"Available columns: {df.columns}"
    )

def normalize_timestamp(column_name):
    return F.to_timestamp(F.col(column_name).cast("string"))

def chronological_split_boundaries(row_count, calibration_fraction, test_fraction):
    if row_count < 3:
        raise ValueError("At least three chronological rows are required")
    test_rows = max(1, int(row_count * test_fraction))
    calibration_rows = max(1, int(row_count * calibration_fraction))
    train_rows = row_count - calibration_rows - test_rows
    if train_rows < 1:
        raise ValueError("Chronological split leaves no training rows")
    return train_rows, train_rows + calibration_rows

ensure_database(GOLD_DB)
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow experiment '{EXPERIMENT_NAME}' ready")

def _normalize_ml_type(data_type):
    value = data_type.replace("bigint", "long").replace("integer", "int")
    return value


def validate_ml_output(frame, table_name):
    contract = ML_OUTPUT_CONTRACTS[table_name]
    expected = tuple((name, data_type) for name, data_type, _ in contract)
    actual = tuple(
        (field.name, _normalize_ml_type(field.dataType.simpleString()))
        for field in frame.schema.fields
    )
    if actual != expected:
        raise RuntimeError(
            f"ML output schema mismatch for {table_name}: expected={expected}, actual={actual}"
        )
    non_nullable = tuple(name for name, _, nullable in contract if not nullable)
    null_flags = frame.agg(
        *(F.max(F.col(name).isNull().cast("int")).alias(name) for name in non_nullable)
    ).first().asDict()
    null_columns = [name for name in non_nullable if null_flags.get(name)]
    if null_columns:
        raise RuntimeError(
            f"ML output {table_name} has nulls in required columns: {null_columns}"
        )
    return frame



---
## Part 1: Data Preparation

Calculate dwell times from truck arrivals and departures.

In [ ]:
print("=" * 60)
print("PREPARING DWELL TIME DATA")
print("=" * 60)

for lifecycle_table in (TRUCK_ARRIVED_TABLE, TRUCK_DEPARTED_TABLE):
    if not bronze_exists(lifecycle_table):
        raise RuntimeError(
            f"{LAKEHOUSE_NAME}.{BRONZE_SCHEMA}.{lifecycle_table} is required; "
            "the completed-only Silver lifecycle cannot represent open arrivals"
        )

df_truck_arrivals = read_bronze(TRUCK_ARRIVED_TABLE)
df_truck_departures = read_bronze(TRUCK_DEPARTED_TABLE)
print(f"Bronze arrivals: {df_truck_arrivals.count()}")
print(f"Bronze departures: {df_truck_departures.count()}")

In [ ]:
# Bronze preserves unmatched arrivals; Silver fact_truck_moves contains only completed pairs.
arrival_rank_window = Window.partitionBy("shipment_id").orderBy(
    F.col("arrived_ts").asc_nulls_last(),
    F.col("arrival_ingest_ts").asc_nulls_last(),
)
df_arrived = (
    df_truck_arrivals
    .select(
        F.col("shipment_id").cast("string").alias("shipment_id"),
        F.regexp_extract(F.col("truck_id"), r"(\d+)$", 1).cast("long").alias("truck_id"),
        F.col("store_id").cast("long").alias("store_id"),
        F.col("dc_id").cast("long").alias("dc_id"),
        normalize_timestamp("arrival_time").alias("arrived_ts"),
        normalize_timestamp("ingest_timestamp").alias("arrival_ingest_ts"),
    )
    .filter(F.col("shipment_id").isNotNull() & F.col("arrived_ts").isNotNull())
    .withColumn("arrival_rank", F.row_number().over(arrival_rank_window))
    .filter(F.col("arrival_rank") == 1)
    .drop("arrival_rank", "arrival_ingest_ts")
    .cache()
)

df_departure_events = (
    df_truck_departures
    .select(
        F.col("shipment_id").cast("string").alias("shipment_id"),
        normalize_timestamp("departure_time").alias("departed_ts"),
    )
    .filter(
        F.col("shipment_id").isNotNull()
        & F.col("departed_ts").isNotNull()
    )
    .groupBy("shipment_id")
    .agg(F.max("departed_ts").alias("departed_ts"))
)

df_departed = (
    df_arrived.alias("arrival")
    .join(df_departure_events.alias("departure"), on="shipment_id", how="inner")
    .select(
        "shipment_id",
        F.col("arrival.truck_id").alias("truck_id"),
        F.col("arrival.store_id").alias("store_id"),
        F.col("arrival.dc_id").alias("dc_id"),
        F.col("arrival.arrived_ts").alias("arrived_ts"),
        F.col("departure.departed_ts").alias("departed_ts"),
    )
    .filter(F.col("departed_ts") > F.col("arrived_ts"))
    .cache()
)

df_dwell = (
    df_departed
    .withColumn(
        "dwell_minutes",
        (F.unix_timestamp("departed_ts") - F.unix_timestamp("arrived_ts")) / 60,
    )
    .withColumn("label_available_ts", F.col("departed_ts"))
    .filter(
        (F.col("dwell_minutes") > 0)
        & (F.col("dwell_minutes") < F.lit(MAX_VALID_DWELL_MINUTES))
    )
    .cache()
)

dwell_count = df_dwell.count()
if dwell_count == 0:
    raise RuntimeError("No valid completed dwell records are available for training")

print(f"\nARRIVED event candidates (open and completed): {df_arrived.count()}")
print(f"Completed shipments with valid dwell: {dwell_count}")
df_dwell.select(
    F.min("dwell_minutes").alias("min_dwell"),
    F.avg("dwell_minutes").alias("avg_dwell"),
    F.max("dwell_minutes").alias("max_dwell"),
    F.stddev("dwell_minutes").alias("stddev_dwell"),
).show()

---
## Part 2: Feature Engineering

Create temporal, location, and historical features.

In [ ]:
print("=" * 60)
print("ARRIVAL-KNOWN FEATURE ENGINEERING")
print("=" * 60)

def add_arrival_known_features(df_input):
    return (
        df_input
        .withColumn("arrival_hour", F.hour("arrived_ts"))
        .withColumn("arrival_day_of_week", F.dayofweek("arrived_ts"))
        .withColumn(
            "is_weekend",
            F.when(F.col("arrival_day_of_week").isin([1, 7]), 1).otherwise(0),
        )
        .withColumn("arrival_date", F.to_date("arrived_ts"))
        .withColumn(
            "location_type",
            F.when(F.col("store_id").isNotNull(), "STORE").otherwise("DC"),
        )
        .withColumn(
            "location_id",
            F.when(F.col("store_id").isNotNull(), F.col("store_id")).otherwise(F.col("dc_id")),
        )
    )

df_features = add_arrival_known_features(df_dwell)
print(f"Arrival-known features created: {df_features.count()} completed records")

In [ ]:
# Load dimensions used by both training and open-shipment inference.
print("\nEnriching with dimension data...")
df_trucks_raw = read_silver(DIM_TRUCKS_TABLE)
truck_id_col = resolve_table_column(df_trucks_raw, DIM_TRUCKS_TABLE, "truck_id", "id", "ID")
truck_refrigeration_col = resolve_table_column(df_trucks_raw, DIM_TRUCKS_TABLE, "refrigeration", "Refrigeration")
truck_home_dc_col = resolve_table_column(df_trucks_raw, DIM_TRUCKS_TABLE, "dc_id", "DCID")
df_trucks = df_trucks_raw.select(
    F.col(truck_id_col).alias("truck_id"),
    F.col(truck_refrigeration_col).cast("double").alias("truck_refrigerated"),
    F.col(truck_home_dc_col).cast("double").alias("truck_home_dc_id"),
)

df_stores_raw = read_silver(DIM_STORES_TABLE)
store_id_col = resolve_table_column(df_stores_raw, DIM_STORES_TABLE, "store_id", "id", "ID")
store_geography_col = resolve_table_column(df_stores_raw, DIM_STORES_TABLE, "geography_id", "GeographyID")
df_stores = df_stores_raw.select(
    F.col(store_id_col).alias("store_id"),
    F.col(store_geography_col).cast("double").alias("store_geography_id"),
)

df_dcs_raw = read_silver(DIM_DCS_TABLE)
dc_id_col = resolve_table_column(df_dcs_raw, DIM_DCS_TABLE, "dc_id", "id", "ID")
dc_geography_col = resolve_table_column(df_dcs_raw, DIM_DCS_TABLE, "geography_id", "GeographyID")
df_dcs = df_dcs_raw.select(
    F.col(dc_id_col).alias("dc_id"),
    F.col(dc_geography_col).cast("double").alias("dc_geography_id"),
)

def enrich_arrival_known_features(df_input):
    return (
        df_input
        .join(df_trucks, on="truck_id", how="left")
        .join(df_stores, on="store_id", how="left")
        .join(df_dcs, on="dc_id", how="left")
        .withColumn("truck_refrigerated", F.coalesce(F.col("truck_refrigerated"), F.lit(0.0)))
        .withColumn("truck_home_dc_id", F.coalesce(F.col("truck_home_dc_id"), F.lit(-1.0)))
        .withColumn(
            "location_geography_id",
            F.coalesce(F.col("store_geography_id"), F.col("dc_geography_id"), F.lit(-1.0)),
        )
        .withColumn("location_id", F.col("location_id").cast("double"))
    )

df_features = enrich_arrival_known_features(df_features)
print("Dimension enrichment complete.")

In [ ]:
# Historical targets are available only after an earlier shipment departed.
print("\nPreparing chronological historical features...")
base_numeric_feature_cols = [
    "arrival_hour", "arrival_day_of_week", "is_weekend", "truck_refrigerated",
    "truck_home_dc_id", "location_id", "location_geography_id",
]
historical_feature_cols = [
    "location_avg_dwell", "location_shipment_count", "hour_avg_dwell",
]
numeric_feature_cols = base_numeric_feature_cols + historical_feature_cols
target_col = "dwell_minutes"
assembler_input_cols = numeric_feature_cols + ["location_type_idx"]

completed_keep_cols = (
    ["shipment_id", "arrived_ts", "departed_ts", "label_available_ts", "location_type"]
    + base_numeric_feature_cols + [target_col]
)
df_model_base = (
    df_features.select(completed_keep_cols)
    .na.drop(subset=completed_keep_cols)
    .dropDuplicates(["shipment_id"])
    .cache()
)
model_base_count = df_model_base.count()
if model_base_count == 0:
    raise RuntimeError("No completed records remain after base feature preparation")

def attach_prior_historical_features(df_targets, df_completed_history):
    history = df_completed_history.select(
        F.col("shipment_id").alias("history_shipment_id"),
        F.col("departed_ts").alias("history_available_ts"),
        F.col("location_id").alias("history_location_id"),
        F.col("location_type").alias("history_location_type"),
        F.col("arrival_hour").alias("history_arrival_hour"),
        F.col(target_col).alias("history_dwell_minutes"),
    )
    target_history = (
        df_targets.alias("target")
        .join(
            history.alias("history"),
            F.col("history.history_available_ts") < F.col("target.arrived_ts"),
            how="left",
        )
    )
    same_location = (
        (F.col("history.history_location_id") == F.col("target.location_id"))
        & (F.col("history.history_location_type") == F.col("target.location_type"))
    )
    same_hour = F.col("history.history_arrival_hour") == F.col("target.arrival_hour")
    prior_stats = (
        target_history
        .groupBy(F.col("target.shipment_id").alias("shipment_id"))
        .agg(
            F.avg("history.history_dwell_minutes").alias("prior_global_avg_dwell"),
            F.count("history.history_dwell_minutes").cast("double").alias("prior_history_count"),
            F.avg(
                F.when(same_location, F.col("history.history_dwell_minutes"))
            ).alias("location_avg_dwell"),
            F.count(
                F.when(same_location, F.col("history.history_dwell_minutes"))
            ).cast("double").alias("location_shipment_count"),
            F.avg(
                F.when(same_hour, F.col("history.history_dwell_minutes"))
            ).alias("hour_avg_dwell"),
        )
    )
    return (
        df_targets
        .join(prior_stats, on="shipment_id", how="left")
        .withColumn(
            "location_avg_dwell",
            F.coalesce(F.col("location_avg_dwell"), F.col("prior_global_avg_dwell")),
        )
        .withColumn(
            "location_shipment_count",
            F.coalesce(F.col("location_shipment_count"), F.lit(0.0)),
        )
        .withColumn(
            "hour_avg_dwell",
            F.coalesce(F.col("hour_avg_dwell"), F.col("prior_global_avg_dwell")),
        )
    )

df_model = (
    attach_prior_historical_features(df_model_base, df_model_base)
    .filter(F.col("prior_history_count") > 0)
    .na.drop(subset=numeric_feature_cols + [target_col])
    .cache()
)
model_count = df_model.count()
if model_count < 3:
    raise RuntimeError("At least three leakage-safe chronological records are required")
print(f"Leakage-safe chronological feature rows: {model_count}")

---
## Part 3: Model Training

Train a distributed Spark ML GBT regressor for point prediction and derive
empirical prediction intervals from residual quantiles.

In [ ]:
print("=" * 60)
print("CHRONOLOGICAL MODEL TRAINING")
print("=" * 60)

train_end_row, calibration_end_row = chronological_split_boundaries(
    model_count, CALIBRATION_SIZE, TEST_SIZE
)
split_order_window = Window.orderBy(F.col("label_available_ts"), F.col("shipment_id"))
df_chronological = df_model.withColumn(
    "chronological_row", F.row_number().over(split_order_window)
).cache()

cutoff_rows = (
    df_chronological
    .filter(F.col("chronological_row").isin([train_end_row, calibration_end_row]))
    .select("chronological_row", "label_available_ts")
    .collect()
)
split_cutoffs = {row["chronological_row"]: row["label_available_ts"] for row in cutoff_rows}
train_label_cutoff = split_cutoffs[train_end_row]
calibration_label_cutoff = split_cutoffs[calibration_end_row]
calibration_start = train_label_cutoff + timedelta(hours=SPLIT_PURGE_HOURS)
test_start = calibration_label_cutoff + timedelta(hours=SPLIT_PURGE_HOURS)

df_train = (
    df_chronological
    .filter(F.col("label_available_ts") <= F.lit(train_label_cutoff))
    .drop("chronological_row").cache()
)
df_calibration = (
    df_chronological
    .filter(
        (F.col("label_available_ts") > F.lit(calibration_start))
        & (F.col("label_available_ts") <= F.lit(calibration_label_cutoff))
    )
    .drop("chronological_row").cache()
)
df_test = (
    df_chronological.filter(F.col("label_available_ts") > F.lit(test_start))
    .drop("chronological_row").cache()
)

train_count = df_train.count()
calibration_count = df_calibration.count()
test_count = df_test.count()
if min(train_count, calibration_count, test_count) == 0:
    raise RuntimeError("Label-availability split is empty after departure-time purge")

indexer = StringIndexer(
    inputCol="location_type", outputCol="location_type_idx", handleInvalid="keep"
)
assembler = VectorAssembler(
    inputCols=assembler_input_cols, outputCol="features", handleInvalid="error"
)
gbt = GBTRegressor(
    featuresCol="features",
    labelCol=target_col,
    predictionCol="predicted_dwell_minutes",
    maxIter=GBT_MAX_ITER,
    maxDepth=GBT_MAX_DEPTH,
    stepSize=GBT_STEP_SIZE,
    subsamplingRate=GBT_SUBSAMPLING_RATE,
    maxBins=GBT_MAX_BINS,
    seed=RANDOM_SEED,
)
point_pipeline = Pipeline(stages=[indexer, assembler, gbt])

print(f"Training set (label-available earliest): {train_count} samples")
print(f"Calibration set after {SPLIT_PURGE_HOURS}h purge: {calibration_count} samples")
print(f"Test set after {SPLIT_PURGE_HOURS}h purge: {test_count} samples")
print(f"Features ({len(assembler_input_cols)}): {', '.join(assembler_input_cols)}")

In [ ]:
print("\nTraining distributed Spark ML GBT model...")

lower_residual_offset = None
upper_residual_offset = None

def add_prediction_intervals(df):
    if lower_residual_offset is None or upper_residual_offset is None:
        raise RuntimeError("Prediction intervals are not calibrated yet")
    return (
        df
        .withColumn(
            "lower_bound_minutes",
            F.greatest(
                F.lit(0.0),
                F.col("predicted_dwell_minutes") + F.lit(lower_residual_offset),
            ),
        )
        .withColumn(
            "upper_bound_minutes",
            F.greatest(
                F.col("lower_bound_minutes"),
                F.col("predicted_dwell_minutes") + F.lit(upper_residual_offset),
            ),
        )
    )

with mlflow.start_run(
    run_name=f"{RUN_NAME_PREFIX}_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M')}"
) as run:
    mlflow.log_params({
        "algorithm": "gbt_regressor_sparkml_chronological",
        "bronze_schema": BRONZE_SCHEMA,
        "truck_arrived_table": TRUCK_ARRIVED_TABLE,
        "truck_departed_table": TRUCK_DEPARTED_TABLE,
        "output_table": OUTPUT_TABLE,
        "max_valid_dwell_minutes": MAX_VALID_DWELL_MINUTES,
        "gbt_max_iter": GBT_MAX_ITER,
        "gbt_max_depth": GBT_MAX_DEPTH,
        "gbt_step_size": GBT_STEP_SIZE,
        "gbt_subsampling_rate": GBT_SUBSAMPLING_RATE,
        "gbt_max_bins": GBT_MAX_BINS,
        "interval_coverage": INTERVAL_COVERAGE,
        "feature_count": len(assembler_input_cols),
        "partition_strategy": "chronological_train_calibration_test",
        "train_rows": train_count,
        "calibration_rows": calibration_count,
        "test_rows": test_count,
    })

    model_point = point_pipeline.fit(df_train)
    print("  Point prediction model trained on earliest partition.")

    # Residual interval offsets come only from the held-out middle partition.
    df_calibration_pred = model_point.transform(df_calibration).cache()
    calibration_residuals_df = df_calibration_pred.select(
        (F.col(target_col) - F.col("predicted_dwell_minutes")).alias("residual")
    ).filter(F.col("residual").isNotNull())
    residual_quantiles = calibration_residuals_df.approxQuantile(
        "residual", [LOWER_RESIDUAL_QUANTILE, UPPER_RESIDUAL_QUANTILE], 0.01
    )
    if len(residual_quantiles) != 2:
        raise RuntimeError("Unable to compute held-out calibration residual quantiles")
    lower_residual_offset, upper_residual_offset = residual_quantiles
    print(
        f"  Held-out residual offsets at {LOWER_RESIDUAL_QUANTILE:.2f}/{UPPER_RESIDUAL_QUANTILE:.2f}: "
        f"{lower_residual_offset:.2f}, {upper_residual_offset:.2f}"
    )

    # The latest partition remains untouched until final evaluation.
    df_test_pred = add_prediction_intervals(model_point.transform(df_test)).cache()
    evaluator_mae = RegressionEvaluator(
        labelCol=target_col, predictionCol="predicted_dwell_minutes", metricName="mae"
    )
    evaluator_rmse = RegressionEvaluator(
        labelCol=target_col, predictionCol="predicted_dwell_minutes", metricName="rmse"
    )
    mae = evaluator_mae.evaluate(df_test_pred)
    rmse = evaluator_rmse.evaluate(df_test_pred)
    mape = (
        df_test_pred.filter(F.col(target_col) > 0)
        .select(
            F.avg(
                F.abs(F.col(target_col) - F.col("predicted_dwell_minutes")) / F.col(target_col)
            ).alias("mape")
        )
        .first()["mape"]
        or 0.0
    )
    interval_metrics = df_test_pred.select(
        F.avg(
            F.when(
                (F.col(target_col) >= F.col("lower_bound_minutes"))
                & (F.col(target_col) <= F.col("upper_bound_minutes")),
                F.lit(1.0),
            ).otherwise(F.lit(0.0))
        ).alias("interval_coverage"),
        F.avg(F.col("upper_bound_minutes") - F.col("lower_bound_minutes")).alias("avg_interval_width"),
    ).first()
    actual_interval_coverage = interval_metrics["interval_coverage"] or 0.0
    avg_interval_width = interval_metrics["avg_interval_width"] or 0.0

    mlflow.log_metrics({
        "mae": round(mae, 2),
        "mape": round(mape, 4),
        "rmse": round(rmse, 2),
        "interval_coverage": round(actual_interval_coverage, 4),
        "avg_interval_width": round(avg_interval_width, 2),
    })
    mlflow.spark.log_model(model_point, MODEL_ARTIFACT_NAME)

    print(f"\nMLflow run: {run.info.run_id}")
    print("\nLatest Test Partition Performance:")
    print(f"  MAE:  {mae:.2f} minutes (Target: < {TARGET_MAE} min)")
    print(f"  MAPE: {mape*100:.2f}% (Target: < {TARGET_MAPE*100}%)")
    print(f"  RMSE: {rmse:.2f} minutes")
    print(f"  Interval Coverage: {actual_interval_coverage*100:.2f}%")
    print(f"  Avg Interval Width: {avg_interval_width:.2f} minutes")

    mae_pass = mae < TARGET_MAE
    mape_pass = mape < TARGET_MAPE
    print(f"  MAE Target:  {'PASS' if mae_pass else 'FAIL'}")
    print(f"  MAPE Target: {'PASS' if mape_pass else 'FAIL'}")

---
## Part 4: Open-Shipment Inference

Infer only ARRIVED shipments with no matched departure/completion record. The
published schema contains arrival-known context and predictions only.

In [ ]:
print("=" * 60)
print("GENERATING OPEN-SHIPMENT PREDICTIONS")
print("=" * 60)

# A shipment is inference-eligible only while its ARRIVED event is unmatched.
df_open_arrivals = (
    df_arrived
    .join(df_departed.select("shipment_id").distinct(), on="shipment_id", how="left_anti")
    .cache()
)
open_arrival_count = df_open_arrivals.count()

df_open_features = enrich_arrival_known_features(
    add_arrival_known_features(df_open_arrivals)
)
inference_base_cols = (
    ["shipment_id", "arrived_ts", "location_type"] + base_numeric_feature_cols
)
df_open_base = (
    df_open_features
    .select(inference_base_cols)
    .na.drop(subset=inference_base_cols)
    .dropDuplicates(["shipment_id"])
)

df_inference_model = (
    attach_prior_historical_features(df_open_base, df_model_base)
    .filter(F.col("prior_history_count") > 0)
    .na.drop(subset=numeric_feature_cols)
    .cache()
)
inference_ready_count = df_inference_model.count()
skipped_without_history_count = max(open_arrival_count - inference_ready_count, 0)

print(
    f"Open ARRIVED shipments: {open_arrival_count}; inference-ready: {inference_ready_count}; "
    f"skipped without earlier completed history: {skipped_without_history_count}"
)
if inference_ready_count == 0:
    raise RuntimeError(
        "No open arrivals have sufficient arrival-known history; existing output is retained"
    )

df_open_pred = (
    add_prediction_intervals(model_point.transform(df_inference_model))
    .withColumn("prediction_timestamp", F.current_timestamp())
    .withColumn("model_run_id", F.lit(run.info.run_id))
    .withColumn("interval_coverage_target", F.lit(INTERVAL_COVERAGE))
    .withColumn("prediction_status", F.lit("OPEN_ARRIVAL_EXPERIMENTAL"))
)

inference_output_cols = [
    "shipment_id",
    "arrived_ts",
    "location_type",
    "arrival_hour",
    "arrival_day_of_week",
    "predicted_dwell_minutes",
    "lower_bound_minutes",
    "upper_bound_minutes",
    "interval_coverage_target",
    "model_run_id",
    "prediction_status",
    "prediction_timestamp",
]
df_predictions = df_open_pred.select(inference_output_cols)

print(f"Generated {df_predictions.count()} open-shipment predictions.")
if inference_ready_count > 0:
    df_predictions.show(10, truncate=False)

In [ ]:
print("Saving open-shipment predictions to Gold...")
df_predictions = validate_ml_output(df_predictions, "dwell_predictions")
save_gold(df_predictions, OUTPUT_TABLE)
print(f"\nPredictions saved to {OUTPUT_TABLE}")

---
## Summary

In [ ]:
print("\n" + "=" * 60)
print("DELIVERY TIME PREDICTION COMPLETE")
print("=" * 60)

print("\nChronological Evaluation:")
print(f"  Train rows: {train_count}")
print(f"  Calibration rows: {calibration_count}")
print(f"  Test rows: {test_count}")
print(f"  MAE:  {mae:.2f} minutes")
print(f"  MAPE: {mape*100:.2f}%")
print(f"  RMSE: {rmse:.2f} minutes")
print(f"  Test interval coverage: {actual_interval_coverage*100:.2f}%")
print(f"  Average interval width: {avg_interval_width:.2f} minutes")

print("\nOpen-Shipment Inference:")
print(f"  Open ARRIVED shipments: {open_arrival_count}")
print(f"  Published predictions: {df_predictions.count()}")
print(f"  Skipped without prior history: {skipped_without_history_count}")
print(f"  Output: {LAKEHOUSE_NAME}.{GOLD_DB}.{OUTPUT_TABLE}")
print("  Output intentionally excludes departures, actual dwell, and prediction errors.")

print(f"\nMLflow experiment: {EXPERIMENT_NAME}")
print(f"Model logged: {MODEL_ARTIFACT_NAME}")
print("Experimental output requires validation and monitoring before operational use.")